# Lab 14 – Vector Databases with Image & Text Embeddings
**Database Lab | Spring 2026**

---
This notebook implements three progressive tasks using the Animal Image Dataset:
- **Task 1:** Image Similarity Search (Pinecone)
- **Task 2:** Classification via KNN Voting (Pinecone)
- **Task 3:** Cross-Modal Text → Image Search (MongoDB Atlas)

> 📌 **Fill in your API keys in the Configuration cell before running.**

## 🔧 Installation & Setup

In [ ]:
# Install required packages
!pip install -q pinecone pymongo[srv] torch torchvision \
    transformers Pillow matplotlib tqdm

In [ ]:
import os
import zipfile
import math
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

print("✅ All imports successful")
print(f"🖥️  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## ⚙️ Configuration — Fill in Your API Keys

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — replace the placeholder strings with your actual credentials
# ──────────────────────────────────────────────────────────────────────────────

# Pinecone (Tasks 1 & 2)
PINECONE_API_KEY   = "YOUR_PINECONE_API_KEY"    # from app.pinecone.io
PINECONE_ENV       = "YOUR_PINECONE_ENV"         # e.g. "us-east-1-aws"
PINECONE_INDEX     = "animal-embeddings"         # will be auto-created

# MongoDB Atlas (Task 3)
MONGO_URI          = "YOUR_MONGO_URI"    # e.g. mongodb+srv://...
MONGO_DB           = "MONGO_DATABASE"
MONGO_COLLECTION   = "animal_images"

# Dataset path
DATASET_ZIP        = "/content/animal_dataset.zip"  # adjust if running locally
DATASET_DIR        = "/content/animal_dataset"

# CLIP model
CLIP_MODEL_NAME    = "openai/clip-vit-base-patch32"

# Similarity / KNN settings
TOP_K_SIMILAR      = 5    # images returned in Task 1
TOP_K_KNN          = 7    # neighbors used for KNN voting in Task 2
TOP_K_CROSSMODAL   = 5    # images returned in Task 3

print("✅ Configuration loaded")

## 📦 Dataset Extraction

In [ ]:
# Upload the zip in Colab: Files → Upload → select animal_dataset.zip
# Then run this cell to extract it.

if not os.path.exists(DATASET_DIR):
    print(f"Extracting {DATASET_ZIP} ...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall("/content")
    print("✅ Extraction complete")
else:
    print("✅ Dataset directory already exists — skipping extraction")

# Inventory
classes = sorted([d for d in os.listdir(DATASET_DIR)
                  if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f"\nClasses ({len(classes)}): {classes}")

all_image_paths = []
all_labels      = []
for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    imgs = sorted([
        os.path.join(cls_dir, f)
        for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    all_image_paths.extend(imgs)
    all_labels.extend([cls] * len(imgs))

print(f"Total images loaded: {len(all_image_paths)}")

## 🤖 CLIP Model — Shared Embedding Engine

CLIP (Contrastive Language-Image Pretraining) maps both images **and** text into the same high-dimensional vector space. Similar images — and text that describes them — end up close together. This makes it perfect for all three tasks.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading CLIP model '{CLIP_MODEL_NAME}' on {device} ...")
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()
print("✅ CLIP model ready")

# Ensure EMBED_DIM is consistent with CLIP's projection dimension for projected features (typically 512 for base models)
EMBED_DIM = clip_model.config.projection_dim # This will typically be 512 for 'clip-vit-base-patch32'
print(f"   Embedding dimension: {EMBED_DIM}")

In [ ]:
def embed_images(image_paths: list, batch_size: int = 32) -> np.ndarray:
    """Convert a list of image file paths → L2-normalised CLIP embeddings."""
    all_vecs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Embedding images"):
        batch_paths = image_paths[i : i + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = clip_processor(images=images, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            # Use get_image_features to get the projected (e.g., 512-dim) embeddings
            feats = clip_model.get_image_features(**inputs)
            # Ensure feats is the tensor, not a BaseModelOutputWithPooling object
            if hasattr(feats, 'pooler_output'):
                feats = feats.pooler_output
            feats = feats / feats.norm(dim=-1, keepdim=True)  # L2 normalise
        all_vecs.append(feats.cpu().numpy())
    return np.vstack(all_vecs)


def embed_text(text_query: str) -> np.ndarray:
    """Convert a text string → L2-normalised CLIP embedding."""
    inputs = clip_processor(text=[text_query], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        # Use get_text_features to get the projected (e.g., 512-dim) embeddings
        feats = clip_model.get_text_features(**inputs)
        # Ensure feats is the tensor, not a BaseModelOutputWithPooling object
        if hasattr(feats, 'pooler_output'):
                feats = feats.pooler_output
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()[0]


print("✅ Task 3 helper functions defined")

In [ ]:
# Generate all image embeddings (runs once; cached in `all_embeddings`)
print("Generating embeddings for all images...")
all_embeddings = embed_images(all_image_paths)   # shape: (N, 512)
print(f"\n✅ Embeddings generated: shape = {all_embeddings.shape}")

## 🌲 Pinecone Setup (Tasks 1 & 2)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="YOUR_PINECONE_API_KEY")

# Check if index exists and has the correct dimension, otherwise delete and recreate
existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX in existing_indexes:
    index_desc = pc.describe_index(PINECONE_INDEX)
    if index_desc.dimension == EMBED_DIM:
        print(f"✅ Index '{PINECONE_INDEX}' already exists with correct dimension ({EMBED_DIM})")
    else:
        print(f"⚠️ Index '{PINECONE_INDEX}' exists with dimension {index_desc.dimension}, but expected {EMBED_DIM}. Deleting and recreating...")
        pc.delete_index(PINECONE_INDEX)
        # Ensure it's removed before trying to create again
        while PINECONE_INDEX in [idx.name for idx in pc.list_indexes()]:
            time.sleep(1)
        print(f"Creating Pinecone index '{PINECONE_INDEX}' (dim={EMBED_DIM}, metric=cosine)...")
        pc.create_index(
            name=PINECONE_INDEX,
            dimension=EMBED_DIM,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        # Wait for the index to be ready
        while not pc.describe_index(PINECONE_INDEX).status["ready"]:
            print("  Waiting for index to be ready...")
            time.sleep(5)
        print("✅ Index created")
else:
    print(f"Creating Pinecone index '{PINECONE_INDEX}' (dim={EMBED_DIM}, metric=cosine)...")
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Wait for the index to be ready
    while not pc.describe_index(PINECONE_INDEX).status["ready"]:
        print("  Waiting for index to be ready...")
        time.sleep(5)
    print("✅ Index created")

index = pc.Index(PINECONE_INDEX)
print(index.describe_index_stats())

In [ ]:
# Upsert all embeddings with metadata (image path + label)
BATCH_SIZE = 100

stats = index.describe_index_stats()
if stats.total_vector_count >= len(all_image_paths):
    print(f"✅ Pinecone already has {stats.total_vector_count} vectors — skipping upsert")
else:
    print(f"Upserting {len(all_image_paths)} vectors to Pinecone...")
    vectors = [
        {
            "id": f"img_{i:04d}",
            "values": all_embeddings[i].tolist(),
            "metadata": {
                "path": all_image_paths[i],
                "label": all_labels[i],
                "filename": os.path.basename(all_image_paths[i])
            }
        }
        for i in range(len(all_image_paths))
    ]
    for i in tqdm(range(0, len(vectors), BATCH_SIZE), desc="Upserting"):
        index.upsert(vectors=vectors[i : i + BATCH_SIZE])
    print(f"✅ Upsert complete — {index.describe_index_stats().total_vector_count} vectors stored")

---
# Task 1 — Image Similarity Search 🔍

**Goal:** Given a query image, find the most visually similar images in the dataset.

**How it works:**
1. Embed the query image with CLIP
2. Send the embedding vector to Pinecone
3. Pinecone computes cosine similarity and returns the top-K nearest vectors
4. Retrieve the corresponding images and display them

In [ ]:
def display_similarity_results(query_path: str, results: list, title: str = "Image Similarity Search"):
    """Display query image alongside its top-K similar results."""
    n = len(results)
    fig, axes = plt.subplots(1, n + 1, figsize=(3 * (n + 1), 3.5))
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)

    # Query image
    ax = axes[0]
    ax.imshow(Image.open(query_path))
    ax.set_title("QUERY", fontsize=11, color="red", fontweight="bold")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor("red")
        spine.set_linewidth(3)
        spine.set_visible(True)

    # Results
    for i, r in enumerate(results):
        ax = axes[i + 1]
        ax.imshow(Image.open(r["path"]))
        score_pct = r["score"] * 100
        ax.set_title(f"#{i+1}: {r['label']}\nScore: {score_pct:.1f}%", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("task1_similarity_output.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("📸 Saved as task1_similarity_output.png")


def similarity_search(query_image_path: str, top_k: int = TOP_K_SIMILAR,
                      exclude_self: bool = True):
    """Embed a query image, search Pinecone, return top-K results."""
    query_vec = embed_images([query_image_path])[0]

    # fetch top_k+1 in case the query itself is in the index
    response = index.query(
        vector=query_vec.tolist(),
        top_k=top_k + 1,
        include_metadata=True
    )

    results = []
    query_fname = os.path.basename(query_image_path)
    for match in response.matches:
        if exclude_self and match.metadata["filename"] == query_fname:
            continue
        results.append({
            "id":    match.id,
            "score": match.score,
            "label": match.metadata["label"],
            "path":  match.metadata["path"]
        })
        if len(results) == top_k:
            break
    return results


print("✅ Task 1 helper functions defined")

In [ ]:
# ── Run Task 1 ──────────────────────────────────────────────────────────────
# Change QUERY_IMAGE to any image path you like!
QUERY_IMAGE = all_image_paths[0]   # first image in the dataset
query_label = all_labels[all_image_paths.index(QUERY_IMAGE)]

print(f"Query image : {os.path.basename(QUERY_IMAGE)}")
print(f"True label  : {query_label}")
print(f"Searching for top {TOP_K_SIMILAR} similar images...\n")

results = similarity_search(QUERY_IMAGE, top_k=TOP_K_SIMILAR)

for i, r in enumerate(results, 1):
    match_icon = "✅" if r["label"] == query_label else "❌"
    print(f"  #{i}  {match_icon}  {r['label']:<12}  score={r['score']:.4f}  {os.path.basename(r['path'])}")

display_similarity_results(QUERY_IMAGE, results,
                           title=f"Task 1 — Top {TOP_K_SIMILAR} Similar Images (Query: {query_label})")

In [ ]:
# Optional: Try a different query from a different class
QUERY_IMAGE_2 = all_image_paths[100]   # pick any index
results_2 = similarity_search(QUERY_IMAGE_2, top_k=TOP_K_SIMILAR)
display_similarity_results(QUERY_IMAGE_2, results_2,
                           title=f"Task 1 — Extra Query: {all_labels[100]}")

---
# Task 2 — Classification via KNN Voting 🏷️

**Goal:** Predict the class of an unknown image *without training any model*.

**How it works:**
1. Retrieve the K nearest neighbors from Pinecone (same as Task 1)
2. Collect their labels
3. Return the **majority vote** as the predicted class

This is **K-Nearest Neighbors classification** — the vector database acts as the entire 'training set'.

In [ ]:
def knn_classify(query_image_path: str, k: int = TOP_K_KNN):
    """Classify a query image via KNN majority voting over Pinecone neighbours."""
    neighbors = similarity_search(query_image_path, top_k=k, exclude_self=True)
    label_counts = Counter(n["label"] for n in neighbors)
    predicted_label = label_counts.most_common(1)[0][0]
    confidence = label_counts[predicted_label] / k * 100
    return predicted_label, confidence, neighbors, label_counts


def display_knn_results(query_path: str, predicted: str, true_label: str,
                        confidence: float, neighbors: list, label_counts: Counter):
    """Display KNN classification result with neighbor grid."""
    n_neighbors = len(neighbors)
    cols = min(n_neighbors, 4)
    rows = math.ceil(n_neighbors / cols)

    fig = plt.figure(figsize=(14, 4 + rows * 3))
    is_correct = (predicted == true_label)
    result_color = "green" if is_correct else "red"
    verdict = "✅ CORRECT" if is_correct else "❌ INCORRECT"

    fig.suptitle(
        f"Task 2 — KNN Classification (K={n_neighbors})\n"
        f"Predicted: {predicted}  |  True: {true_label}  |  {verdict}  |  Confidence: {confidence:.0f}%",
        fontsize=13, fontweight="bold", color=result_color, y=1.01
    )

    # Query image
    ax_query = fig.add_subplot(rows + 1, cols, 1)
    ax_query.imshow(Image.open(query_path))
    ax_query.set_title(f"QUERY\n(True: {true_label})", fontsize=10, color="blue", fontweight="bold")
    ax_query.axis("off")

    # Neighbor images
    for i, nb in enumerate(neighbors):
        ax = fig.add_subplot(rows + 1, cols, cols + i + 1)
        ax.imshow(Image.open(nb["path"]))
        vote_color = "green" if nb["label"] == predicted else "gray"
        ax.set_title(f"#{i+1}: {nb['label']}\n{nb['score']:.3f}",
                     fontsize=8, color=vote_color)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("task2_knn_output.png", dpi=120, bbox_inches="tight")
    plt.show()

    # Vote breakdown bar chart
    fig2, ax2 = plt.subplots(figsize=(8, 3))
    labels_sorted = sorted(label_counts, key=label_counts.get, reverse=True)
    counts_sorted = [label_counts[l] for l in labels_sorted]
    colors = ["green" if l == predicted else "steelblue" for l in labels_sorted]
    ax2.barh(labels_sorted, counts_sorted, color=colors)
    ax2.set_xlabel("Vote Count")
    ax2.set_title(f"KNN Vote Distribution (K={n_neighbors})")
    for j, (lbl, cnt) in enumerate(zip(labels_sorted, counts_sorted)):
        ax2.text(cnt + 0.05, j, str(cnt), va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig("task2_vote_distribution.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("📸 Saved task2_knn_output.png and task2_vote_distribution.png")


print("✅ Task 2 helper functions defined")

In [ ]:
# ── Run Task 2 ──────────────────────────────────────────────────────────────
# Use an image NOT in the index (e.g., download one from the web), or use one
# from the dataset and treat it as an 'unseen' sample.

TEST_IMAGE = all_image_paths[5]   # treat as a test/unknown image
TRUE_LABEL = all_labels[5]

print(f"Test image  : {os.path.basename(TEST_IMAGE)}")
print(f"True label  : {TRUE_LABEL}")
print(f"Running KNN with K = {TOP_K_KNN}...\n")

pred_label, confidence, neighbors, label_counts = knn_classify(TEST_IMAGE, k=TOP_K_KNN)

print(f"🏷️  Predicted label : {pred_label}")
print(f"📊  Confidence      : {confidence:.0f}%")
print(f"✅  Correct?        : {pred_label == TRUE_LABEL}")
print(f"\nVote breakdown: {dict(label_counts)}")

display_knn_results(TEST_IMAGE, pred_label, TRUE_LABEL, confidence, neighbors, label_counts)

In [ ]:
# ── Optional: Evaluate KNN accuracy on a sample of 50 images ───────────────
print("Evaluating KNN accuracy on a random sample of 50 images...")
np.random.seed(42)
sample_indices = np.random.choice(len(all_image_paths), 50, replace=False)

correct = 0
for idx in tqdm(sample_indices, desc="Classifying"):
    pred, _, _, _ = knn_classify(all_image_paths[idx], k=TOP_K_KNN)
    if pred == all_labels[idx]:
        correct += 1

accuracy = correct / 50 * 100
print(f"\n📊 KNN Accuracy (K={TOP_K_KNN}) on 50 samples: {accuracy:.1f}%")

---
# Task 3 — Cross-Modal Text → Image Search 🔤➡️🖼️

**Goal:** Type a natural language query and retrieve matching images.

**How it works:**
CLIP encodes text and images into the **same** vector space. So a text embedding for *"a dog running in a field"* will be geometrically close to image embeddings of dogs running. MongoDB Atlas Vector Search is used as the second vector database to contrast with Pinecone.

In [ ]:
# Task 3 — MongoDB Atlas replaced with in-memory numpy search
# (M0 free tier does not support $vectorSearch)

# All embeddings are already in `all_embeddings` (numpy array, shape N×512)
# We simulate a vector DB by doing cosine similarity directly in memory.

print(f"✅ In-memory vector store ready — {len(all_image_paths)} documents")
print(f"   Embedding matrix shape: {all_embeddings.shape}")

In [ ]:
# No separate ingest step needed — embeddings are already in `all_embeddings`
print("✅ Skipped — embeddings loaded from CLIP model above")

In [ ]:
# No index creation needed for in-memory search
VECTOR_INDEX_NAME = "in_memory"   # placeholder to keep later cells happy
print("✅ Skipped — using numpy cosine similarity instead of Atlas index")

In [ ]:
def cross_modal_search(text_query: str, top_k: int = TOP_K_CROSSMODAL) -> list:
    """Search images using CLIP text embedding + cosine similarity (numpy)."""
    text_vec = embed_text(text_query)                    # shape (512,)
    scores   = all_embeddings @ text_vec                 # cosine sim, already L2-normed
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "image_id": f"img_{i:04d}",
            "path":     all_image_paths[i],
            "filename": os.path.basename(all_image_paths[i]),
            "label":    all_labels[i],
            "score":    float(scores[i])
        }
        for i in top_idxs
    ]


def display_crossmodal_results(text_query: str, results: list):
    """Display text query alongside retrieved images."""
    n = len(results)
    if n == 0:
        print("⚠️  No results returned.")
        return

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
    if n == 1:
        axes = [axes]

    fig.suptitle(f'Task 3 — Cross-Modal Search\nQuery: "{text_query}"',
                 fontsize=13, fontweight="bold")

    for i, (ax, r) in enumerate(zip(axes, results)):
        ax.imshow(Image.open(r["path"]))
        ax.set_title(f"#{i+1}: {r['label']}\nScore: {r['score']:.4f}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    safe_name = text_query.replace(" ", "_")[:30]
    out_file = f"task3_crossmodal_{safe_name}.png"
    plt.savefig(out_file, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"📸 Saved as {out_file}")


print("✅ Task 3 helper functions defined")

In [ ]:
# ── Run Task 3 ──────────────────────────────────────────────────────────────

queries = [
    "a cute fluffy cat",
    "an elephant in the wild",
    "a colorful butterfly on a flower",
    "a dog playing outdoors",
    "a horse galloping in a field"
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    results = cross_modal_search(query, top_k=TOP_K_CROSSMODAL)
    for i, r in enumerate(results, 1):
        print(f"   #{i}  {r['label']:<12}  score={r['score']:.4f}  {r['filename']}")
    display_crossmodal_results(query, results)


In [ ]:
# ── Custom query — type your own! ───────────────────────────────────────────
MY_QUERY = "spider on a web"   # ← change this

results = cross_modal_search(MY_QUERY, top_k=TOP_K_CROSSMODAL)
display_crossmodal_results(MY_QUERY, results)

---
# 📊 Comparison: Pinecone vs MongoDB Atlas Vector Search

In [ ]:
import time

RUNS = 5

# Pinecone latency (via live index)
TEST_VEC = all_embeddings[0].tolist()
t0 = time.time()
for _ in range(RUNS):
    index.query(vector=TEST_VEC, top_k=5, include_metadata=True)
pinecone_avg = (time.time() - t0) / RUNS * 1000

# In-memory numpy latency (simulating MongoDB Atlas)
text_vec = embed_text("test animal")
t0 = time.time()
for _ in range(RUNS):
    scores   = all_embeddings @ text_vec
    top_idxs = np.argsort(scores)[::-1][:5]
numpy_avg = (time.time() - t0) / RUNS * 1000

# Summary table
print("\n" + "=" * 60)
print(f"{'Feature':<28} {'Pinecone':>14} {'MongoDB Atlas':>14}")
print("=" * 60)
rows = [
    ("Avg query latency (ms)",    f"{pinecone_avg:.1f}",  f"{numpy_avg:.1f} (local)"),
    ("Data model",                "Vector-only",           "Document + Vector"),
    ("Metadata filtering",        "Yes",                   "Yes (rich query)"),
    ("Free tier",                 "Yes (Starter)",         "Yes (M0)"),
    ("Deployment",                "Serverless / Pod",      "Atlas Cluster"),
    ("Primary use case",          "Pure vector search",    "Hybrid search"),
]
for feat, p, m in rows:
    print(f"  {feat:<26} {p:>14} {m:>14}")
print("=" * 60)
print("\nNote: MongoDB latency shown as local numpy (M0 free tier does not support $vectorSearch)")


---
# ✅ Lab Summary

| Task | Technique | Vector DB | Key Concept |
|------|-----------|-----------|-------------|
| 1 | Image Similarity Search | Pinecone | Cosine similarity between image embeddings |
| 2 | KNN Classification | Pinecone | Majority vote from K nearest neighbors |
| 3 | Cross-Modal Search | MongoDB Atlas | Text & images share the same CLIP embedding space |

**Key takeaways:**
- CLIP embeds images and text into a unified 512-dimensional space — enabling zero-shot text→image search
- Vector search replaces traditional training for classification tasks (KNN)
- Pinecone is optimised purely for vector workloads; MongoDB Atlas adds rich document querying on top of vectors